# Custom Auto EDA

Building our own automated EDA tool — designed around the best practices and strengths of ydata-profiling, sweetviz, dtale, dataprep, and autoviz.

## Design Philosophy

| Principle | Source | What it means |
|---|---|---|
| **Modular API** | dataprep | Every section is an independent callable — run one or all |
| **Dtype-aware** | autoviz | Numeric, categorical, datetime each get the right analysis |
| **Warnings system** | ydata-profiling | Flag issues automatically without manual inspection |
| **Target analysis** | sweetviz | Optional `target_col` parameter activates feature-vs-target views |
| **Drift detection** | sweetviz | Compare two DataFrames (train vs test) with statistical tests |
| **Full correlations** | ydata-profiling | Pearson for numeric, Cramér's V for categorical, point-biserial for mixed |
| **Code transparency** | dtale | Each function prints what it is computing — no black box |
| **Shareable output** | ydata-profiling / sweetviz | Final `generate_report()` writes a self-contained HTML file |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
# Rich synthetic dataset — designed to exercise all 10 sections
n = 1200

df = pd.DataFrame({
    # Numeric — normal
    'age':           np.random.randint(18, 72, n).astype(float),
    'score':         np.random.uniform(1, 10, n).round(2),
    'tenure_days':   np.random.randint(0, 1825, n).astype(float),

    # Numeric — right-skewed (log-normal)
    'income':        np.random.exponential(52000, n).round(2),
    'revenue':       np.random.exponential(3500, n).round(2),

    # Numeric — correlated with income
    'spend':         None,

    # Categorical — low cardinality
    'city':          np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Pune'], n),
    'tier':          np.random.choice(['Bronze', 'Silver', 'Gold', 'Platinum'], n,
                                       p=[0.4, 0.3, 0.2, 0.1]),
    'channel':       np.random.choice(['Online', 'Retail', 'B2B'], n),

    # Categorical — high cardinality (will trigger a warning)
    'product_code':  [f'SKU-{i:04d}' for i in np.random.randint(0, 900, n)],

    # Datetime
    'signup_date':   pd.date_range('2020-01-01', periods=n, freq='7h'),

    # Binary classification target
    'churn':         np.random.choice([0, 1], n, p=[0.78, 0.22]),

    # Near-constant column (will trigger a warning)
    'flag':          np.random.choice([0, 1], n, p=[0.99, 0.01]),
})

# Correlated feature
df['spend'] = (df['income'] * np.random.uniform(0.05, 0.30, n)).round(2)

# Inject missing values (realistic pattern)
df.loc[np.random.choice(df.index, 90),  'age']     = np.nan   # ~7.5%
df.loc[np.random.choice(df.index, 140), 'income']  = np.nan   # ~12%
df.loc[np.random.choice(df.index, 50),  'city']    = np.nan   # ~4%
df.loc[np.random.choice(df.index, 30),  'revenue'] = np.nan   # ~2.5%

# Inject a few outliers into revenue
df.loc[np.random.choice(df.index, 8), 'revenue'] = np.random.uniform(80000, 120000, 8)

print(f"Dataset ready: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
df.head()

---
## Next Steps — Building Plan

Each section below is an independent module. Work through them sequentially, adding code cells after each header. The function signatures are defined here as the API contract — implement the body section by section.

---

### Section 1 — Dataset Overview
**Inspired by:** ydata-profiling overview tab  
**Best practice:** Surface shape, memory, duplicates and missing summary in a single glance — the first thing any analyst needs.

**What to build:**
- Row count, column count, total cells
- Memory usage: total (MB) + per-column breakdown
- Dtype counts: how many numeric / categorical / datetime / boolean
- Duplicate row count and %
- Total missing cells count and overall % missing
- 5-row preview (head)

```python
def overview(df: pd.DataFrame) -> dict:
    ...
```

---

### Section 2 — Column Profiler
**Inspired by:** ydata-profiling per-column stats, dataprep `plot()`  
**Best practice:** Every column gets the same baseline facts plus dtype-specific statistics — one consistent table, no manual if/else by the analyst.

**What to build:**
- All columns: `dtype`, `null_count`, `null_pct`, `unique_count`, `unique_pct`, `top_value`
- **Numeric only:** `mean`, `median`, `std`, `min`, `p25`, `p75`, `max`, `skewness`, `kurtosis`, `IQR`
- **Categorical only:** top-5 value counts + cardinality ratio
- **Datetime only:** `min_date`, `max_date`, `date_range_days`, estimated frequency

```python
def column_profile(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

---

### Section 3 — Missing Value Analysis
**Inspired by:** ydata-profiling missing tab, dataprep `plot_missing()`  
**Best practice:** Missing counts alone are not enough — co-occurrence patterns reveal whether missingness is random (MCAR) or structured (MAR/MNAR).

**What to build:**
- Sorted horizontal bar chart: missing % per column (descending)
- Missingness heatmap: row × column binary matrix (white=present, black=missing)
- Co-occurrence matrix: for each pair of columns, what % of rows are missing in both
- Missingness correlation: Pearson correlation between the boolean missing-indicator columns

```python
def missing_analysis(df: pd.DataFrame) -> None:
    ...
```

---

### Section 4 — Distribution Analysis
**Inspired by:** autoviz auto-chart selection, ydata-profiling per-column distributions  
**Best practice:** Select the right chart for the right dtype automatically — histograms + KDE for numeric, sorted bar charts for categorical. Flag skewness and suggest transformations.

**What to build:**
- **Numeric:** histogram + KDE overlay; annotate skewness value; if `|skew| > 1` → add note "consider log transform"
- **Categorical:** horizontal bar chart sorted by frequency descending; annotate count on each bar
- **Datetime:** line chart of row count over time (monthly/weekly bins)
- `cols=None` runs all columns; `cols=['income', 'city']` runs a subset

```python
def distribution_analysis(df: pd.DataFrame, cols: list = None) -> None:
    ...
```

---

### Section 5 — Outlier Detection
**Inspired by:** dtale outlier detection UI, ydata-profiling outlier flags  
**Best practice:** Offer two methods (IQR and Z-score) so the analyst can compare. Always show both the statistical summary and the visual — a box plot makes outliers intuitive.

**What to build:**
- **IQR method:** flag values below `Q1 − 1.5×IQR` or above `Q3 + 1.5×IQR`
- **Z-score method:** flag values where `|z| > 3`
- Box plots for all numeric columns with outlier count annotated in the title
- Summary DataFrame: `column`, `method`, `outlier_count`, `outlier_pct`, `lower_bound`, `upper_bound`

```python
def outlier_report(df: pd.DataFrame, method: str = 'iqr') -> pd.DataFrame:
    ...
```

---

### Section 6 — Correlation Analysis
**Inspired by:** ydata-profiling correlations (Pearson + Spearman + Cramér's V), sweetviz association matrix  
**Best practice:** A single correlation matrix that mixes numeric and categorical using the right metric for each pair — not just Pearson, which ignores categoricals entirely.

**What to build:**
- **Numeric ↔ Numeric:** Pearson + Spearman heatmaps side by side
- **Categorical ↔ Categorical:** Cramér's V (normalised chi-square) heatmap
- **Numeric ↔ Categorical:** Point-biserial correlation
- Unified association heatmap combining all three
- Flag pairs where `|r| > 0.90` as highly correlated (multicollinearity risk)

```python
def correlation_analysis(df: pd.DataFrame, threshold: float = 0.90) -> None:
    ...
```

---

### Section 7 — Target Analysis
**Inspired by:** sweetviz target feature analysis  
**Best practice:** The most valuable insight in a supervised ML project. Every feature should be examined through the lens of the target — which features separate the classes / correlate with the outcome?

**What to build:**
- **Classification target:**
  - Numeric features: overlapping KDE per class (churn=0 vs churn=1)
  - Categorical features: grouped % bar chart (what % of each tier churned?)
  - Ranked table: feature → association with target (eta-squared / Cramér's V)
- **Regression target:**
  - Scatter plot + regression line per numeric feature
  - Box plot per categorical feature
  - Ranked table: feature → Pearson r with target
- `task='classification'` or `task='regression'` parameter

```python
def target_analysis(df: pd.DataFrame, target_col: str,
                    task: str = 'classification') -> None:
    ...
```

---

### Section 8 — Data Quality Warnings
**Inspired by:** ydata-profiling warnings system  
**Best practice:** Surface problems automatically without the analyst having to hunt for them. Each warning includes the column name, the issue, and a recommended action.

**What to build — flag the following:**

| Warning | Condition | Recommended action |
|---|---|---|
| **Constant column** | `nunique == 1` | Drop — carries no signal |
| **Near-zero variance** | `std < 0.01 × |mean|` | Investigate — likely encoding issue |
| **High cardinality** | `unique_pct > 50%` for categorical | Consider hashing or grouping |
| **High missingness** | `null_pct > 40%` | Impute carefully or drop |
| **High skewness** | `|skew| > 2` | Log or Box-Cox transform |
| **High correlation** | `|r| > 0.95` between any pair | Check for redundancy / leakage |
| **Duplicate columns** | Two columns with identical values | Drop one |
| **Mixed types** | `dtype == object` but >50% values are numeric strings | Cast to numeric |

```python
def quality_warnings(df: pd.DataFrame) -> list[dict]:
    # Returns: [{'column': ..., 'warning': ..., 'action': ...}, ...]
    ...
```

---

### Section 9 — Train / Test Comparison
**Inspired by:** sweetviz compare mode (its single strongest feature)  
**Best practice:** Distribution drift between train and test is one of the most common silent bugs in ML projects. Use statistical tests, not visual inspection, to detect it reliably.

**What to build:**
- **Numeric columns:** Kolmogorov–Smirnov test (`scipy.stats.ks_2samp`) — `p < 0.05` flags drift
- **Categorical columns:** Chi-square test (`scipy.stats.chi2_contingency`) — `p < 0.05` flags drift
- Missing rate comparison per column (train % vs test %)
- Overlapping KDE plots for drifted numeric columns
- Summary DataFrame: `column`, `dtype`, `test_used`, `statistic`, `p_value`, `drifted`

```python
def compare_splits(train_df: pd.DataFrame,
                   test_df: pd.DataFrame) -> pd.DataFrame:
    ...
```

---

### Section 10 — HTML Report Generator
**Inspired by:** ydata-profiling + sweetviz export — the feature that makes EDA shareable  
**Best practice:** A report that exists only inside the notebook is useless to stakeholders. Embed all plots as base64 images so the HTML is self-contained and requires no server.

**What to build:**
- Call Sections 1–9 in sequence, capturing outputs
- Convert all matplotlib figures to base64 PNG strings (`io.BytesIO` + `base64.b64encode`)
- Render DataFrames as styled HTML tables
- Build an HTML string with:
  - Collapsible `<details>` sections per analysis block
  - A CSS stylesheet (clean, minimal — no external dependencies)
  - A timestamp and dataset metadata in the header
- Write to `output_path` as a single `.html` file

```python
def generate_report(df: pd.DataFrame,
                    output_path: str = 'eda_report.html',
                    target_col: str = None,
                    train_df: pd.DataFrame = None,
                    test_df: pd.DataFrame = None) -> None:
    ...
```

---
## Section 1 — Dataset Overview

---
## Section 2 — Column Profiler

---
## Section 3 — Missing Value Analysis

---
## Section 4 — Distribution Analysis

---
## Section 5 — Outlier Detection

---
## Section 6 — Correlation Analysis

---
## Section 7 — Target Analysis

---
## Section 8 — Data Quality Warnings

---
## Section 9 — Train / Test Comparison

---
## Section 10 — HTML Report Generator